# Gradient maps visualization

### Load model and titles

In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import random

st_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
base_model = st_model._first_module().auto_model
tokenizer = st_model.tokenizer

DATASET = 'Amazon_Sports_and_Outdoors'

ITEM_PATH = f'../../{DATASET}/{DATASET}.item'

df = pd.read_csv(ITEM_PATH, delimiter='\t')
df = df[['item_id:token', 'title:token']]

titles = df['title:token'].astype('str').tolist()

title = random.choice(titles)

2025-10-24 01:10:49.507985: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 01:10:49.551606: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 01:10:50.808656: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Convert title to tokens

In [2]:
inputs = tokenizer(title, return_tensors='pt', add_special_tokens=True)

device = next(base_model.parameters()).device
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)

embeds = base_model.embeddings.word_embeddings(input_ids)
embeds.retain_grad()

### Calculate vanilla saliency map

In [3]:
output = base_model(inputs_embeds=embeds, attention_mask=attention_mask)
hidden_state = output.last_hidden_state

sentence_embedding = hidden_state.mean(dim=1)

target = sentence_embedding.norm()
target.backward()

token_gradients = embeds.grad.norm(dim=-1).squeeze().detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze())
token_importance = [(t, w.item()) for t, w in zip(tokens, token_gradients) if t not in tokenizer.all_special_tokens]

print(f'Original title: {title}')
for token, weight in token_importance:
    print(f'{token}: {weight:.2f}')

Original title: Beartooth Made in USA Universal Fit Brown Color Neoprene Recoil Pad Kit For Mosin Nagant 91/30 M44 M38 M39 Mauser K98 M! Garand Springfield 1903 Remington 700 798 Howa 1500 Weatherby Vanguard Winchester 70 Rifles
bear: 1.16
##tooth: 1.86
made: 0.22
in: 0.15
usa: 0.25
universal: 0.49
fit: 0.24
brown: 0.40
color: 0.23
neo: 0.76
##pre: 0.60
##ne: 0.44
recoil: 1.35
pad: 0.63
kit: 0.42
for: 0.13
mo: 0.42
##sin: 0.57
naga: 0.73
##nt: 0.21
91: 0.43
/: 0.14
30: 0.14
m: 0.12
##44: 0.37
m3: 0.34
##8: 0.16
m3: 0.49
##9: 0.29
ma: 0.15
##user: 0.36
k: 0.15
##9: 0.20
##8: 0.20
m: 0.37
!: 0.58
ga: 0.33
##rand: 0.48
springfield: 0.55
1903: 0.39
remington: 0.48
700: 0.35
79: 0.29
##8: 0.22
how: 0.19
##a: 0.17
1500: 0.27
weather: 0.35
##by: 0.28
vanguard: 0.34
winchester: 0.47
70: 0.29
rifles: 0.41
